# 🎲 Probabilistic Tiny Recursive Models (PTRM)

**Community implementation of [arXiv:2605.19943](https://arxiv.org/abs/2605.19943)**

This notebook demonstrates the complete PTRM pipeline:

1. **Model Download** — Fetch pre-trained TRM checkpoints from HuggingFace
2. **Model Loading** — Instantiate TRM architectures from manifests
3. **TRM Baseline Inference** — Standard deterministic TRM inference (K=1, σ=0)
4. **PTRM Stochastic Inference** — K parallel rollouts with noise injection
5. **Evaluation & Metrics** — pass@K, best-Q@K, mode@K
6. **Visualization** — PCA dynamics, Q-tracking, basin escape, width scaling, noise ablation
7. **TRM vs PTRM Comparison** — Side-by-side analysis

---

### Key Idea

TRM models recursively refine latent states $z_H$ and $z_L$ to solve structured puzzles. PTRM adds noise ($z_H \mathrel{+}= \varepsilon,\; \varepsilon \sim \mathcal{N}(0, \sigma^2 I)$) before each supervision step, runs **K parallel rollouts**, and selects the best solution using the model's own Q-head (**best-Q@K**).

No additional training is required — PTRM is **inference-only**.

## 0. Setup & Imports

In [ ]:
# Install dependencies (run once)
!pip install -q -r requirements.txt

In [ ]:
import os
import sys
import time

# Ensure we're in the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if os.path.basename(PROJECT_ROOT) != 'PTRM':
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# PTRM modules
from inference.checkpoint_loader import (
    load_model_from_manifest,
    list_available_models,
    _add_trm_to_path,
)
from inference.ptrm_inference import PTRMInference, PTRMBatchResult
from evaluation.metrics import (
    compute_metrics_from_result,
    compute_cell_accuracy,
    compute_exact_accuracy,
    format_metrics,
)
from visualization.pca_latent import plot_pca_latent_dynamics
from visualization.q_tracking import plot_q_tracking
from visualization.basin_escape import plot_basin_escape, compute_basin_statistics
from visualization.width_scaling import plot_width_scaling
from visualization.noise_ablation import plot_noise_ablation

# Device selection: CUDA > MPS > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("⚠️ Running on CPU — inference will be slow. GPU recommended for full benchmarks.")

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
## 1. 📥 Model Download

Download pre-trained TRM checkpoints from HuggingFace Hub. Three benchmarks are available:

| Benchmark | Model | Params | HuggingFace Repo |
|:----------|:------|-------:|:-----------------|
| Sudoku-Extreme | TRM-MLP | 5M | alphaXiv/trm-model-sudoku |
| Maze-Hard 30×30 | TRM-Att | 5M | Sanjin2024/TinyRecursiveModel-Maze-Hard |
| ARC-AGI-2 | TRM-Att | 5M | arcprize/trm_arc_prize_verification |

In [ ]:
# Download models (skip if already downloaded)
from scripts.download_models import download_all, download_benchmark

# Download the Sudoku model (smallest dataset, fastest for demo)
print("Downloading Sudoku-Extreme model...")
download_benchmark('sudoku', output_dir='models')
print("\n✅ Model download complete!")

In [ ]:
# List all downloaded models
available = list_available_models('models')
print(f"Available models: {len(available)}")
for m in available:
    print(f"  • {m['benchmark']} ({m['arch_type']}) — manifest: {m['manifest_path']}")

---
## 2. 🏗️ Model Loading

Load a TRM model from its manifest file. The manifest contains:
- Architecture config (hidden_size, H/L cycles, etc.)
- Dataset metadata (vocab_size, seq_len)
- Checkpoint file paths

The loader handles:
- `torch.compile` prefix stripping (`_orig_mod.` → removed)
- Puzzle identifier introspection from weight shapes
- Automatic eval mode

In [ ]:
# Load the Sudoku-Extreme TRM-MLP model
MANIFEST_PATH = 'models/sudoku/manifest.yaml'
BATCH_SIZE = 4  # Small batch for CPU demo

model, model_meta = load_model_from_manifest(
    MANIFEST_PATH,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

print(f"Model type: {model_meta.get('model_type', 'N/A')}")
print(f"Hidden size: {model.inner.config.hidden_size}")
print(f"H-cycles: {model.inner.config.H_cycles}")
print(f"L-cycles: {model.inner.config.L_cycles}")
print(f"Seq len: {model.inner.config.seq_len}")
print(f"Vocab size: {model.inner.config.vocab_size}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 3. 🧩 Create a Synthetic Sudoku Batch

For demonstration, we create a batch of synthetic Sudoku inputs.

In the real Sudoku-Extreme dataset:
- Inputs: 9×9 grid flattened to 81 tokens, clues filled, blanks as 1 (shifted: 0=PAD, 1-9→2-10)
- Labels: Complete solution grid

For this demo, we generate random inputs to exercise the inference pipeline.

In [ ]:
# Create a synthetic batch matching Sudoku format
SEQ_LEN = 81  # 9x9 Sudoku grid
VOCAB_SIZE = 11  # PAD + 10 tokens (shifted digits)

# Random inputs (simulating partially-filled Sudoku grids)
torch.manual_seed(42)
synthetic_inputs = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))

# Random labels (simulating complete solutions)
synthetic_labels = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))

# Puzzle identifiers (all same puzzle type for Sudoku)
puzzle_ids = torch.zeros(BATCH_SIZE, dtype=torch.long)

batch = {
    'inputs': synthetic_inputs,
    'puzzle_identifiers': puzzle_ids,
}

print(f"Batch shape: {synthetic_inputs.shape}")
print(f"Sample input (first 9 tokens = first row): {synthetic_inputs[0, :9].tolist()}")
print(f"Sample label (first 9 tokens = first row): {synthetic_labels[0, :9].tolist()}")

---
## 4. 🔄 TRM Baseline Inference (Deterministic)

Standard TRM inference: **K=1 rollout, σ=0** (no noise).

This is the baseline that PTRM builds upon. With no noise, the model always produces the same output for the same input — it's trapped in a single fixed-point attractor.

In [ ]:
# Create PTRM inference engine
engine = PTRMInference(model, device=DEVICE)

# TRM Baseline: K=1, sigma=0 (deterministic)
print("Running TRM baseline (K=1, D=4, σ=0)...")
t0 = time.time()

trm_result = engine.run(
    batch,
    K=1,           # Single rollout = standard TRM
    D=4,           # 4 supervision steps (reduced for CPU speed)
    sigma=0.0,     # No noise = deterministic
    seed=42,
    collect_trajectories=True,
)

trm_time = time.time() - t0
print(f"  Completed in {trm_time:.1f}s")
print(f"  Predictions shape: {trm_result.all_predictions.shape}")
print(f"  Q-values shape: {trm_result.all_q_values.shape}")
print(f"  Sample prediction (first 9): {trm_result.best_q_predictions[0, :9].tolist()}")

In [ ]:
# Verify determinism: run again with same seed
trm_result_2 = engine.run(batch, K=1, D=4, sigma=0.0, seed=42)

is_identical = torch.equal(trm_result.best_q_predictions, trm_result_2.best_q_predictions)
print(f"Deterministic (same output with same seed): {is_identical}")
print(f"Q-values identical: {torch.equal(trm_result.all_q_values, trm_result_2.all_q_values)}")

---
## 5. 🎲 PTRM Stochastic Inference (K Parallel Rollouts)

Now the key contribution: inject noise into $z_H$ at each supervision step,
run **K parallel rollouts**, and select the best using the Q-head.

**Algorithm 1 (from the paper):**
```
For k = 1..K in parallel:
    Initialize z_k from model's init state
    For t = 1..D:
        z_k.z_H += ε,  ε ~ N(0, σ²I)     # Noise injection
        z_k, y_k = inner(z_k, x)           # One supervision step
    ŷ_k = argmax f_O(y_k)                  # Decode output
    q̂_k = f_Q(y_k)                         # Q-head score
Return ŷ_{k*}, where k* = argmax_k q̂_k     # Best-Q selection
```

In [ ]:
# PTRM: K=10 parallel rollouts with noise
K = 10
D = 4      # Supervision steps (use 16 for full benchmark, 4 for CPU demo)
SIGMA = 0.3

print(f"Running PTRM inference (K={K}, D={D}, σ={SIGMA})...")
t0 = time.time()

ptrm_result = engine.run(
    batch,
    K=K,
    D=D,
    sigma=SIGMA,
    seed=42,
    collect_trajectories=True,  # Collect latent states for visualization
)

ptrm_time = time.time() - t0
print(f"  Completed in {ptrm_time:.1f}s")
print(f"  All predictions: {ptrm_result.all_predictions.shape}  (B, K, seq_len)")
print(f"  All Q-values: {ptrm_result.all_q_values.shape}  (B, K)")
print(f"  Latent trajectories: {ptrm_result.latent_trajectories.shape}  (B, K, D, latent_dim)")
print(f"  Step Q-values: {ptrm_result.step_q_values.shape}  (B, K, D)")

In [ ]:
# Examine the diversity of rollouts
print("\n=== Rollout Diversity (Puzzle 0) ===")
preds_0 = ptrm_result.all_predictions[0]  # (K, seq_len)
q_vals_0 = ptrm_result.all_q_values[0]    # (K,)

# Count unique predictions
unique_preds = torch.unique(preds_0, dim=0)
print(f"  Total rollouts: {K}")
print(f"  Unique predictions: {unique_preds.shape[0]}")
print(f"  Q-value range: [{q_vals_0.min():.3f}, {q_vals_0.max():.3f}]")
print(f"  Q-value std: {q_vals_0.std():.4f}")

# Show per-rollout diversity (first 9 tokens = first row of Sudoku)
print(f"\n  First row predictions per rollout:")
for k in range(min(K, 5)):
    print(f"    Rollout {k}: {preds_0[k, :9].tolist()}  (Q={q_vals_0[k]:.3f})")
if K > 5:
    print(f"    ... ({K-5} more rollouts)")

# Selection results
print(f"\n  Best-Q selection: {ptrm_result.best_q_predictions[0, :9].tolist()}")
print(f"  Mode selection:   {ptrm_result.mode_predictions[0, :9].tolist()}")

---
## 6. 📊 Evaluation & Metrics

Three selection methods:
- **pass@K** — Oracle upper bound: any of K rollouts is correct
- **best-Q@K** — Rollout with highest Q-value (inference-time selectable)
- **mode@K** — Majority vote among K rollouts

In [ ]:
# Compute metrics against synthetic labels
ptrm_metrics = compute_metrics_from_result(
    ptrm_result,
    synthetic_labels.to(DEVICE),
    ignore_id=0,
)

print(format_metrics(ptrm_metrics))

In [ ]:
# Also compute TRM baseline metrics for comparison
trm_metrics = compute_metrics_from_result(
    trm_result,
    synthetic_labels.to(DEVICE),
    ignore_id=0,
)

print("=== TRM Baseline (K=1, σ=0) ===")
print(format_metrics(trm_metrics))
print("\n=== PTRM (K=10, σ=0.3) ===")
print(format_metrics(ptrm_metrics))

# Improvement
cell_improvement = ptrm_metrics.mean_pass_at_k_cell - trm_metrics.mean_pass_at_k_cell
print(f"\n📈 pass@K cell accuracy improvement: {cell_improvement:+.2%}")

---
## 7. 🎨 Visualization Suite

Five visualizations from the paper, demonstrating how PTRM explores latent space.

### 7.1 PCA Latent Dynamics (Figure 1)

2D PCA projection of $z_H[:, 0, :]$ trajectories across D supervision steps.
Each line = one rollout's path through latent space. Color = Q-value (green = high confidence).

With noise, rollouts diverge into different regions of latent space, exploring multiple solution basins.

In [ ]:
# PCA Latent Dynamics — PTRM (σ=0.3)
fig = plot_pca_latent_dynamics(
    ptrm_result.latent_trajectories,
    ptrm_result.all_q_values,
    puzzle_idx=0,
    title=f'PTRM Latent Dynamics (K={K}, D={D}, σ={SIGMA})',
)
plt.show()

In [ ]:
# Compare: TRM baseline (σ=0) — all rollouts collapse to same trajectory
# Run K=5 with sigma=0 to show the contrast
trm_k5 = engine.run(batch, K=5, D=D, sigma=0.0, seed=42, collect_trajectories=True)

fig = plot_pca_latent_dynamics(
    trm_k5.latent_trajectories,
    trm_k5.all_q_values,
    puzzle_idx=0,
    title='TRM Baseline (K=5, D=4, σ=0) — All Trajectories Collapse',
)
plt.show()

### 7.2 Q-Value Tracking

Q-halt logit evolution across supervision steps. Shows how the Q-head's confidence develops over time.

In PTRM, different rollouts develop different confidence levels, with the best-Q rollout (black line) emerging as the model identifies the most promising solution path.

In [ ]:
# Q-value tracking across supervision steps
fig = plot_q_tracking(
    ptrm_result.step_q_values,
    puzzle_idx=0,
    title=f'Q-Value Dynamics (K={K}, D={D}, σ={SIGMA})',
)
plt.show()

### 7.3 Basin Escape Analysis

Left panel: Pairwise L2 distances between rollout endpoints in latent space.
Right panel: Q-value distribution sorted by confidence.

This shows that noise injection creates **meaningfully different** solution trajectories (large pairwise distances) rather than just random perturbation.

In [ ]:
# Basin escape analysis
fig = plot_basin_escape(
    ptrm_result.latent_trajectories,
    ptrm_result.all_q_values,
    puzzle_idx=0,
    title=f'Basin Escape Analysis (K={K}, D={D}, σ={SIGMA})',
)
plt.show()

# Print statistics
stats = compute_basin_statistics(
    ptrm_result.latent_trajectories,
    ptrm_result.all_q_values,
    puzzle_idx=0,
)
print(f"Q-value spread (σ_Q): {stats['q_spread']:.4f}")
print(f"Trajectory divergence: {stats['trajectory_divergence']:.2f}")
print(f"Estimated clusters: {stats['endpoint_cluster_count']}")

### 7.4 Width Scaling Curves

How accuracy improves as we increase K (number of parallel rollouts).
This is the central result of the paper — PTRM achieves significant gains over baseline TRM.

**Note:** This cell runs inference for multiple K values, which may take a few minutes on CPU.

In [ ]:
# Width scaling sweep: vary K
K_values = [1, 2, 3, 5, 8]
D_fixed = 4
sigma_fixed = 0.3

pass_at_k_results = []
bestq_at_k_results = []
mode_at_k_results = []

print("Running width scaling sweep...")
for K_val in tqdm(K_values, desc='Width scaling'):
    result = engine.run(batch, K=K_val, D=D_fixed, sigma=sigma_fixed, seed=42)
    metrics = compute_metrics_from_result(result, synthetic_labels.to(DEVICE), ignore_id=0)
    
    pass_at_k_results.append(metrics.mean_pass_at_k_cell)
    bestq_at_k_results.append(metrics.mean_best_q_at_k_cell)
    mode_at_k_results.append(metrics.mean_mode_at_k_cell)
    
    print(f"  K={K_val:>3d}: pass@K={metrics.mean_pass_at_k_cell:.3f}  "
          f"best-Q@K={metrics.mean_best_q_at_k_cell:.3f}  "
          f"mode@K={metrics.mean_mode_at_k_cell:.3f}")

In [ ]:
# Plot width scaling curves
fig = plot_width_scaling(
    K_values,
    pass_at_k_results,
    bestq_at_k_results,
    mode_at_k_results,
    metric_type='cell',
    baseline=pass_at_k_results[0],  # K=1 as baseline
    baseline_label='TRM Baseline (K=1)',
    title=f'PTRM Width Scaling (D={D_fixed}, σ={sigma_fixed})',
)
plt.show()

### 7.5 Noise Ablation Sweep

How accuracy varies with noise level σ. The paper finds that moderate noise (σ ≈ 0.3) provides the best balance between diversity and quality.

- **σ = 0**: No exploration — stuck in one basin (equivalent to TRM)
- **σ too large**: Noise overwhelms the signal — predictions become random
- **σ ≈ 0.3**: Sweet spot — diverse exploration while maintaining solution quality

In [ ]:
# Noise ablation sweep: vary sigma
sigma_values = [0.0, 0.1, 0.3, 0.5, 1.0]
K_fixed = 5
D_fixed = 4

noise_pass = []
noise_bestq = []
noise_mode = []

print("Running noise ablation sweep...")
for sigma_val in tqdm(sigma_values, desc='Noise ablation'):
    result = engine.run(batch, K=K_fixed, D=D_fixed, sigma=sigma_val, seed=42)
    metrics = compute_metrics_from_result(result, synthetic_labels.to(DEVICE), ignore_id=0)
    
    noise_pass.append(metrics.mean_pass_at_k_cell)
    noise_bestq.append(metrics.mean_best_q_at_k_cell)
    noise_mode.append(metrics.mean_mode_at_k_cell)
    
    print(f"  σ={sigma_val:.1f}: pass@K={metrics.mean_pass_at_k_cell:.3f}  "
          f"best-Q@K={metrics.mean_best_q_at_k_cell:.3f}")

In [ ]:
# Plot noise ablation
fig = plot_noise_ablation(
    sigma_values,
    noise_pass,
    noise_bestq,
    noise_mode,
    K=K_fixed,
    metric_type='cell',
    title=f'Noise Ablation (K={K_fixed}, D={D_fixed})',
)
plt.show()

---
## 8. ⚔️ TRM vs PTRM Comparison

Side-by-side comparison showing why PTRM outperforms standard TRM.

| Property | TRM | PTRM |
|:---------|:----|:-----|
| Rollouts | 1 | K (parallel) |
| Noise | None (σ=0) | σ > 0 injected into z_H |
| Selection | N/A (only one output) | best-Q@K or mode@K |
| Training | Required | None (inference-only) |
| Latent exploration | Single fixed point | Multiple basins |

In [ ]:
# TRM vs PTRM side-by-side comparison
K_compare = 8
D_compare = 4

# TRM: K rollouts but no noise (all identical)
trm_compare = engine.run(
    batch, K=K_compare, D=D_compare, sigma=0.0, seed=42, collect_trajectories=True
)

# PTRM: K rollouts with noise (diverse)
ptrm_compare = engine.run(
    batch, K=K_compare, D=D_compare, sigma=0.3, seed=42, collect_trajectories=True
)

# Count unique predictions
trm_unique = torch.unique(trm_compare.all_predictions[0], dim=0).shape[0]
ptrm_unique = torch.unique(ptrm_compare.all_predictions[0], dim=0).shape[0]

print(f"Unique predictions (K={K_compare}):")
print(f"  TRM  (σ=0.0): {trm_unique} / {K_compare} unique")
print(f"  PTRM (σ=0.3): {ptrm_unique} / {K_compare} unique")

# Q-value diversity
trm_q_std = trm_compare.all_q_values[0].std().item()
ptrm_q_std = ptrm_compare.all_q_values[0].std(correction=0).item()
print(f"\nQ-value diversity (σ_Q):")
print(f"  TRM:  {trm_q_std:.4f}")
print(f"  PTRM: {ptrm_q_std:.4f}")

In [ ]:
# Visual comparison: PCA trajectories side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# TRM: all trajectories collapse
plt.sca(ax1)
fig_trm = plot_pca_latent_dynamics(
    trm_compare.latent_trajectories,
    trm_compare.all_q_values,
    puzzle_idx=0,
    title=f'TRM (K={K_compare}, σ=0) — Collapsed',
)

# PTRM: diverse trajectories
plt.sca(ax2)
fig_ptrm = plot_pca_latent_dynamics(
    ptrm_compare.latent_trajectories,
    ptrm_compare.all_q_values,
    puzzle_idx=0,
    title=f'PTRM (K={K_compare}, σ=0.3) — Diverse',
)

plt.tight_layout()
plt.show()

# Close the individual figures to avoid duplication
plt.close(fig_trm)
plt.close(fig_ptrm)

In [ ]:
# Metrics comparison
trm_m = compute_metrics_from_result(trm_compare, synthetic_labels.to(DEVICE), ignore_id=0)
ptrm_m = compute_metrics_from_result(ptrm_compare, synthetic_labels.to(DEVICE), ignore_id=0)

comparison_data = {
    'Metric': ['pass@K (cell)', 'best-Q@K (cell)', 'mode@K (cell)',
               'pass@K (exact)', 'best-Q@K (exact)', 'mode@K (exact)'],
    'TRM (σ=0)': [
        f"{trm_m.mean_pass_at_k_cell:.2%}",
        f"{trm_m.mean_best_q_at_k_cell:.2%}",
        f"{trm_m.mean_mode_at_k_cell:.2%}",
        f"{trm_m.mean_pass_at_k_exact:.2%}",
        f"{trm_m.mean_best_q_at_k_exact:.2%}",
        f"{trm_m.mean_mode_at_k_exact:.2%}",
    ],
    'PTRM (σ=0.3)': [
        f"{ptrm_m.mean_pass_at_k_cell:.2%}",
        f"{ptrm_m.mean_best_q_at_k_cell:.2%}",
        f"{ptrm_m.mean_mode_at_k_cell:.2%}",
        f"{ptrm_m.mean_pass_at_k_exact:.2%}",
        f"{ptrm_m.mean_best_q_at_k_exact:.2%}",
        f"{ptrm_m.mean_mode_at_k_exact:.2%}",
    ],
}

# Display as formatted table
print(f"{'Metric':<25s} {'TRM (σ=0)':>12s} {'PTRM (σ=0.3)':>14s}")
print('=' * 55)
for i, metric in enumerate(comparison_data['Metric']):
    print(f"{metric:<25s} {comparison_data['TRM (σ=0)'][i]:>12s} {comparison_data['PTRM (σ=0.3)'][i]:>14s}")

---
## 9. 🏆 Paper Results Reference

The paper reports the following results on actual benchmarks (Table 1):

| Benchmark | Model | pass@25 | best-Q@25 | mode@25 |
|:----------|:------|--------:|----------:|--------:|
| Sudoku-Extreme | TRM-MLP | **92.3%** | 65.3% | 63.8% |
| Maze-Hard 30×30 | TRM-Att | **85.2%** | 71.3% | 68.5% |
| ARC-AGI-2 | TRM-Att | **35.4%** | 24.1% | 22.7% |

To replicate these results, run the full benchmark on a CUDA GPU:
```bash
python scripts/download_models.py    # Download all models
python scripts/download_data.py      # Build all datasets
./scripts/run_all_benchmarks.sh      # Run evaluation
```

> **Note:** The demo above uses synthetic data and reduced parameters (small K, D) for speed.
> Real benchmark results require the actual datasets and D=16 (Sudoku/Maze) or D=64 (ARC).

---
## 10. 🛠️ Configuration System

Each benchmark has a YAML configuration file that can be loaded and customized.

In [ ]:
from config.config_loader import load_inference_config, list_available_configs

# List all available configs
configs = list_available_configs('config/inference')
print("Available inference configurations:")
for c in configs:
    print(f"  • {c['name']}: K={c['K']}, D={c['D']}, σ={c['sigma']}")

# Load a config with overrides
config = load_inference_config(
    'config/inference/sudoku_extreme.yaml',
    overrides={'ptrm.K': 50, 'ptrm.sigma': 0.5}
)
print(f"\nLoaded config (with overrides):")
print(f"  K={config.K}, D={config.D}, σ={config.sigma}")
print(f"  Manifest: {config.manifest_path}")
print(f"  Data: {config.data_path}")

---
## Summary

This notebook demonstrated the complete PTRM pipeline:

1. ✅ **Model Download** — HuggingFace checkpoint retrieval with manifest generation
2. ✅ **Model Loading** — Unified loader with compile prefix stripping
3. ✅ **TRM Baseline** — Deterministic inference (K=1, σ=0)
4. ✅ **PTRM Inference** — K parallel stochastic rollouts with Q-head selection
5. ✅ **Metrics** — pass@K, best-Q@K, mode@K evaluation
6. ✅ **Visualization** — PCA dynamics, Q-tracking, basin escape, width scaling, noise ablation
7. ✅ **TRM vs PTRM** — Side-by-side comparison showing diversity and accuracy gains

### Key Takeaway

> PTRM transforms a deterministic TRM model into a probabilistic reasoner **without any additional training** — 
> just noise injection and K-parallel rollouts with Q-head selection at inference time.

For full benchmark replication, see the [README](./README.md) and run:
```bash
./scripts/run_all_benchmarks.sh
```